# IndicTranscribe ASR — inference walkthrough

End-to-end tour of `bodhan_genai.asr`: batch transcription, long-form audio,
language identification, and the offline batch CLI.

**Read [docs/asr/caveats.md](../../docs/asr/caveats.md) before trusting any WER number.**
Two things bite first:

1. the model is **language-conditioned** — a wrong language label produces confidently
   wrong *script*, not obvious garbage;
2. **long audio needs chunking** — whole-file quality collapses past ~60 s.

Requires a GPU and a converted checkpoint directory (`pip install -e ".[asr-infer]"`).

In [ ]:
import torch

from bodhan_genai.asr import IndicASREngine

MODEL_DIR = "/path/to/indic-transcribe-hf"  # converted HF checkpoint dir
AUDIO = ["sample_hi_1.wav", "sample_hi_2.wav"]  # 16 kHz or any rate; resampled internally
LANG = "hi"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={device}")

## 1. Load the engine

The checkpoint's safetensors are fp32 (the master copy); pass `dtype=torch.bfloat16`
at load time for inference. bf16 is **WER-neutral**, not an improvement: measured at
−0.10% on 512 utterances and 0.00% on sub-1 s clips, with per-language deltas moving
in both directions.

In [ ]:
engine = IndicASREngine(MODEL_DIR, device=device, dtype=torch.bfloat16)

print(f"vocab={engine.model.config.vocab_size} d_model={engine.model.config.d_model}")
print(f"encoder layers={engine.model.config.encoder_layers}")
print(f"decoder layers={engine.model.config.decoder_layers}")
print(f"feature sample rate={engine.fe.sample_rate}")

## 2. Batch transcription

**A batch must be single-language**, because the 10-token prompt encodes the language
in two of its slots. Group by language before batching (the CLI does this for you).

Duration-sort within a batch too: the encoder pads to the batch maximum, so mixing a
3 s clip with a 40 s clip wastes most of the batch.

In [ ]:
texts = engine.transcribe_batch(AUDIO, lang=LANG)
for path, text in zip(AUDIO, texts, strict=True):
    print(f"{path}\n  {text}\n")

### What the frozen prompt implies about the output

Every utterance is prompted identically (only the language slots change), so the
output style is fixed and intentional:

- **native script** — English and mathematics are transliterated, not kept in Latin;
- numbers are spelled as **words**, not digits;
- **punctuation is emitted** (the legacy pipeline passed `punc=False`, but NeMo
  silently dropped that kwarg — the port reproduces the behaviour production
  actually had).

The native-script behaviour is the single biggest source of apparent WER when
references keep English in Latin — see caveats §3.1.

In [ ]:
prompt = engine.tokenizer.encode_prompt(LANG)
print(f"prompt ids: {prompt}")
print(f"pieces:     {engine.tokenizer.ids_to_pieces(prompt)}")

## 3. Other input styles

Paths, in-memory waveforms, and a pre-collated tensor all take the identical
padding / mono / resample path, so switching between them cannot move the text.

In [ ]:
wavs = [engine.load_audio(p) for p in AUDIO]  # 1-D tensors at 16 kHz
print([tuple(w.shape) for w in wavs])

from_waveforms = engine.transcribe_batch(wavs, lang=LANG)
assert from_waveforms == texts, "input style must not change the output"
print("waveform input matches path input")

## 4. Long-form audio

Past ~60 s the decoder emits EOS early and degrades into repetition — a 221 s file
produced 149 words against a 450-word reference. `transcribe_long` splits on
silences and joins the chunk transcripts.

`chunk_above` is a **threshold, not a switch**: audio at or below it is transcribed
whole, because chunking short audio measurably hurts (32.50% vs 30.15% WER at 15 s).

In [ ]:
LONG_AUDIO = "long_interview.wav"

text, chunks = engine.transcribe_long(
    LONG_AUDIO,
    lang=LANG,
    chunk_above=45.0,  # measured knee
    chunk_min=15.0,  # best-WER window from the sweep
    chunk_max=25.0,
    return_chunks=True,
)

print(f"{len(chunks)} chunk(s)\n")
for start_s, end_s, chunk_text in chunks:
    print(f"[{start_s:7.2f} - {end_s:7.2f}] {chunk_text[:80]}...")

### Inspecting the segmentation without a model

The chunker is pure torch, so you can tune the window on CPU before spending GPU time.

In [ ]:
from bodhan_genai.asr.engine import ChunkConfig, split_points

wav = engine.load_audio(LONG_AUDIO)
sr = engine.fe.sample_rate
print(f"duration: {wav.numel() / sr:.1f}s\n")

for lo, hi in [(10, 15), (15, 25), (20, 30)]:
    segs = split_points(wav, sr, ChunkConfig(min_chunk=lo, max_chunk=hi))
    lengths = [(b - a) / sr for a, b in segs]
    mean_len = sum(lengths) / len(lengths)
    print(f"{lo}-{hi}s window -> {len(segs)} chunks, mean {mean_len:.1f}s")

## 5. Language identification

The transcription path has no LID — you must supply a label. But the checkpoint can
identify the language itself: feeding only the 3-token language-independent prompt
head and reading the next position asks the model which language token belongs in the
`source_lang` slot. That is **one decoder step**, not a second transcription.

96.9% top-1 agreement with the NeMo detector — but it **cannot reliably separate
hi/ur**, so resolve that pair from metadata.

In [ ]:
for path, top in zip(AUDIO, engine.detect_language(AUDIO), strict=True):
    guesses = ", ".join(f"{lang}={p:.3f}" for lang, p in top[:3])
    print(f"{path}\n  {guesses}")

In [ ]:
# Detect, then transcribe with the detected label.
detected = engine.detect_language([AUDIO[0]])[0][0][0]
print(f"detected: {detected}")
print(engine.transcribe_batch([AUDIO[0]], lang=detected)[0])

## 6. Offline batch CLI

For corpora rather than a handful of files: sharded, resumable, one process per GPU.

```bash
python -m bodhan_genai.asr.inference.transcribe \
    --manifest data.jsonl --model-dir /path/to/indic-transcribe-hf --out-dir out/ \
    --chunk-above 45

# 8 GPUs
MODEL_DIR=/path/to/indic-transcribe-hf NUM_SHARDS=8 \
    scripts/asr/infer.sh --manifest data.jsonl --out-dir out/
```

Manifest rows are JSON objects with an audio path and a language:
```json
{"audio_path": "/audio/0001.wav", "language": "hi"}
```

Resume is keyed on **manifest row index**, not on an id field — in the corpus this
port was built against, 84k rows carried only 48k unique keys, so key-based resume
silently dropped duplicates.

Budget **4–8 CPU cores per GPU**: with 2 CPUs for 8 GPUs, per-shard encode time went
from 18 s to 437 s.

## Next

- [docs/asr/model.md](../../docs/asr/model.md) — architecture, parity results, deviations
- [docs/asr/caveats.md](../../docs/asr/caveats.md) — settings, the duration/quality curve, scoring pitfalls
- [docs/asr/usage.md](../../docs/asr/usage.md) — API and CLI reference